# 12.7 功率相差很大，取 log 有什麼效果？


四個頻譜格的功率為 `[0,10⁻¹⁰,10⁻⁴,1]`。若直接用同一亮度尺度，前三格可能幾乎都看不見。取自然對數能壓縮跨度，但零不能直接取 log，所以先設最小值 10⁻⁸。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

power = torch.tensor([0.0, 1e-10, 1e-4, 1.0])
floor = 1e-8
compressed = power.clamp(min=floor).log()
scaled = (4 * power).clamp(min=floor).log()
print("log功率", [round(x, 2) for x in compressed.tolist()])
print("放大後差", [round(x, 2) for x in (scaled - compressed).tolist()])
print("皆為有限值", bool(compressed.isfinite().all()))

結果約為 `[-18.42,-18.42,-9.21,0]`，全都是有限值。前兩格低於下限，都先變成 10⁻⁸；這些差異已被捨棄。負值只表示原功率小於 1，不表示能量為負。

功率乘 4 後，未被下限夾住的兩格增加 `ln(4)≈1.39`；前兩格仍在下限，差為 0。乘法變成加法，有助於讓大尺度變化比較平緩。若波形振幅乘 2，功率才乘 4，不能混用振幅與功率的倍數。

把頻率功率合成 mel 頻帶後再取 log，就得到 log-mel 特徵。這是固定算式整理聲音，而不是成熟辨識模型提供的隱藏答案。它仍保留每一時間框，以便後續入口學會利用局部變化。

下限與後續正規化要一致，尤其比較安靜片段時；過大的下限會讓不同弱聲音得到相同數值。練習把下限改為 10⁻¹²，零仍需替代，但 10⁻¹⁰ 不再被夾住。說明「避免無限值」與「保留弱聲音差異」為何需要同時考慮。

<details>
<summary>回顧與查證</summary>

可回顧：[W.4機率與log](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.4)、[12.6的mel功率](https://birdhackor.github.io/tiny-perceptron-vlm/12.6.html)。

</details>
